# Supermarket Sales Analysis
## Data Analytics Project

**Objective:** Analyze supermarket sales transactions to identify useful patterns in products, branches, categories, customers, payment methods, and ratings.

**Dataset:** `SUPER MARKET DATA.pdf` — 500 sales transactions.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "SUPER MARKET DATA.pdf"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

COLUMNS = ["Invoice ID", "Date", "Branch", "City", "Customer Type", "Gender",
           "Product", "Category", "Quantity", "Unit Price", "Payment", "Rating", "Sales"]

print("Dataset:", PDF_PATH)

## 1. Load and extract the dataset

In [ ]:
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    text = "\n".join(page.extract_text(x_tolerance=1, y_tolerance=3) or "" for page in pdf.pages)
text = text.replace("\f", "\n")

pattern = re.compile(
    r"^(INV\d+)\s+(\d{4}-\d{2}-\d{2})\s+"
    r"([A-D])\s+(Jaipur|Delhi|Mumbai|Bengaluru)\s+"
    r"(Member|Normal)\s+(Male|Female)\s+"
    r"(.+?)\s+(Dairy|Grocery|Personal Care|Fruits|Vegetables|Snacks|Beverages|Bakery)\s+"
    r"(\d+)\s+([\d.]+)\s+(UPI|Net Banking|Card|Cash)\s+"
    r"([\d.]+)\s+([\d.]+)\s*$", re.MULTILINE)

rows = pattern.findall(text)
df = pd.DataFrame(rows, columns=COLUMNS)
if len(df) != 500:
    raise ValueError(f"Expected 500 transactions, extracted {len(df)}")

df["Date"] = pd.to_datetime(df["Date"])
for col in ["Quantity", "Unit Price", "Rating", "Sales"]:
    df[col] = pd.to_numeric(df[col])

print("Rows extracted:", len(df))
display(df.head())

## 2. Data cleaning and validation

In [ ]:
df = df.drop_duplicates(subset=["Invoice ID"]).copy()
df["Calculated Sales"] = (df["Quantity"] * df["Unit Price"]).round(2)
df["Sales Check"] = (df["Calculated Sales"] - df["Sales"]).abs() < 0.01

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate invoice IDs:", int(df["Invoice ID"].duplicated().sum()))
print("Sales formula checks passed:", int(df["Sales Check"].sum()), "/", len(df))

df.to_csv(OUTPUT_DIR / "supermarket_sales_cleaned.csv", index=False)

## 3. Descriptive statistics

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Transactions", "Total Sales", "Average Transaction", "Average Rating"],
    "Value": [len(df), df["Sales"].sum(), df["Sales"].mean(), df["Rating"].mean()]
})
display(summary)

## 4. Product analysis

In [ ]:
product_sales = df.groupby("Product", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
display(product_sales)
print("Top product:", product_sales.iloc[0]["Product"])
print("Top product sales: ₹{:,.2f}".format(product_sales.iloc[0]["Sales"]))

plt.figure(figsize=(9, 5))
top10 = product_sales.head(10).sort_values("Sales")
plt.barh(top10["Product"], top10["Sales"])
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales (₹)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()
product_sales.to_csv(OUTPUT_DIR / "sales_by_product.csv", index=False)

## 5. Branch analysis

In [ ]:
branch_sales = df.groupby(["Branch", "City"], as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
display(branch_sales)

plt.figure(figsize=(8, 5))
plot_data = branch_sales.sort_values("Sales")
plt.barh(plot_data["Branch"].astype(str) + " (" + plot_data["City"] + ")", plot_data["Sales"])
plt.title("Sales by Branch")
plt.xlabel("Sales (₹)")
plt.ylabel("Branch")
plt.tight_layout()
plt.show()
branch_sales.to_csv(OUTPUT_DIR / "sales_by_branch.csv", index=False)

## 6. Category analysis

In [ ]:
category_sales = df.groupby("Category", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
display(category_sales)

plt.figure(figsize=(9, 5))
plot_data = category_sales.sort_values("Sales")
plt.barh(plot_data["Category"], plot_data["Sales"])
plt.title("Sales by Category")
plt.xlabel("Sales (₹)")
plt.ylabel("Category")
plt.tight_layout()
plt.show()
category_sales.to_csv(OUTPUT_DIR / "sales_by_category.csv", index=False)

## 7. Payment and customer analysis

In [ ]:
payment_usage = df["Payment"].value_counts().rename_axis("Payment").reset_index(name="Transactions")
customer_type_avg = df.groupby("Customer Type", as_index=False)["Sales"].mean()
display(payment_usage)
display(customer_type_avg)

plt.figure(figsize=(8, 5))
plot_data = payment_usage.sort_values("Transactions")
plt.barh(plot_data["Payment"], plot_data["Transactions"])
plt.title("Payment Method Usage")
plt.xlabel("Transactions")
plt.ylabel("Payment Method")
plt.tight_layout()
plt.show()

payment_usage.to_csv(OUTPUT_DIR / "payment_counts.csv", index=False)
customer_type_avg.to_csv(OUTPUT_DIR / "customer_type_average.csv", index=False)

## 8. Monthly sales trend and final insights

In [ ]:
monthly_sales = (
    df.assign(Month=df["Date"].dt.to_period("M").astype(str))
      .groupby("Month", as_index=False)["Sales"].sum()
)
display(monthly_sales)

plt.figure(figsize=(9, 5))
plt.plot(monthly_sales["Month"], monthly_sales["Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
monthly_sales.to_csv(OUTPUT_DIR / "monthly_sales.csv", index=False)

print("KEY FINDINGS")
print("- Highest-sales product: {} — ₹{:,.2f}".format(product_sales.iloc[0]["Product"], product_sales.iloc[0]["Sales"]))
print("- Best branch: Branch {} ({}) — ₹{:,.2f}".format(branch_sales.iloc[0]["Branch"], branch_sales.iloc[0]["City"], branch_sales.iloc[0]["Sales"]))
print("- Highest-sales category: {} — ₹{:,.2f}".format(category_sales.iloc[0]["Category"], category_sales.iloc[0]["Sales"]))
print("- Most-used payment method: {} — {} transactions".format(payment_usage.iloc[0]["Payment"], payment_usage.iloc[0]["Transactions"]))
print("- Member average transaction: ₹{:,.2f}".format(customer_type_avg.loc[customer_type_avg["Customer Type"] == "Member", "Sales"].iloc[0]))
print("- Normal average transaction: ₹{:,.2f}".format(customer_type_avg.loc[customer_type_avg["Customer Type"] == "Normal", "Sales"].iloc[0]))
print("- Average customer rating: {:.2f}/5".format(df["Rating"].mean()))

## Conclusion and business recommendations

1. Maintain sufficient stock for high-selling products and categories.
2. Study the practices contributing to Branch C's strong performance.
3. Continue supporting UPI payments because it is the most-used method.
4. Use customer feedback and ratings to improve service quality.
5. Compare Member and Normal customer spending when planning membership offers.